# LeetCode 669 — Trim a Binary Search Tree bằng Python

Link đề: https://leetcode.com/problems/trim-a-binary-search-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Search Tree, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def trimBST(self, root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
        ...
```

Ghi chú: notebook tự dựng `TreeNode`, hàm `build_tree` (dựng cây từ list kiểu LeetCode, ví dụ `[3,0,4,null,2,null,null,1]`) và `tree_to_list` (xuất cây ra list cùng kiểu). Các ví dụ "tự đặt" (cây một nút bị cắt hết cho kết quả rỗng `[]`, v.v.) đã được kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một **cây tìm kiếm nhị phân (BST)** và đoạn `[low, high]`, **cắt tỉa** cây để mọi nút còn lại đều có giá trị trong đoạn đó, **giữ nguyên cấu trúc tương đối** (nút nào vốn là hậu duệ của nút nào thì vẫn thế), rồi trả về **gốc mới** (gốc có thể đổi).
2. Nắm các công cụ Python: định nghĩa `TreeNode`, dựng/xuất cây từ list có `None`, đệ quy trả về "gốc của cây con đã xử lý", sửa con trỏ tại chỗ (`node.left = node.left.right`), ngăn xếp bằng `list`, và duyệt tiền thứ tự (preorder).
3. Đi từ **thu mọi giá trị trong đoạn rồi chèn lại vào BST mới** (`O(n · h)`) → **đệ quy theo 3 trường hợp** (lời giải nộp, `O(n)`) → **lặp: đưa gốc vào đoạn rồi tỉa hai "xương sống" trái/phải** (`O(n)` thời gian, `O(1)` phụ) → **duyệt tiền thứ tự có cắt nhánh rồi dựng lại cây `O(n)` bằng ngăn xếp**.
4. Hiểu mấu chốt tính chất BST: nếu `root.val > high` thì **cả cây con phải** (toàn giá trị lớn hơn `root.val`) cũng vượt `high`, nên bỏ hẳn gốc và cây phải, chỉ cần xét cây con trái; tương tự khi `root.val < low`.
5. Tránh các lỗi kinh điển: không gán lại kết quả đệ quy cho `root.left/right`, đi nhầm hướng khi gốc ra ngoài đoạn, dựng lại cây "cân bằng" làm **mất cấu trúc** mà đề yêu cầu giữ, quên bước đưa gốc vào đoạn trong bản lặp, và đệ quy quá sâu với cây lệch tới `10^4` nút.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một **cây tìm kiếm nhị phân** và hai biên `low`, `high`. Hãy **cắt tỉa** cây sao cho mọi phần tử còn lại đều nằm trong `[low, high]`. Việc cắt tỉa **không được** làm thay đổi cấu trúc tương đối của các phần tử còn lại (tức mọi hậu duệ của một nút vẫn là hậu duệ của nó). Có thể chứng minh đáp án là **duy nhất**.

Trả về **gốc của cây đã cắt tỉa**. Lưu ý gốc có thể thay đổi tùy theo hai biên.

**Ví dụ 1**

- Input: `root = [1,0,2], low = 1, high = 2`
- Output: `[1,null,2]`
- Giải thích: nút `0 < 1` bị bỏ; nút `2` vẫn là con phải của `1`.

**Ví dụ 2**

- Input: `root = [3,0,4,null,2,null,null,1], low = 1, high = 3`
- Output: `[3,2,null,1]`
- Giải thích: nút `0 < 1` bị bỏ, con phải của nó (`2`, kèm con trái `1`) được nâng lên làm con trái của `3`; nút `4 > 3` bị bỏ.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [5], low = 1, high = 2`
- Output: `[]` (cây rỗng, vì `5` nằm ngoài đoạn).

**Ràng buộc:**

- Số nút của cây trong khoảng `[1, 10^4]`.
- `0 <= Node.val <= 10^4`.
- Giá trị của các nút là **duy nhất**.
- `root` được đảm bảo là BST hợp lệ.
- `0 <= low <= high <= 10^4`.

Chú ý: kết quả có thể là `None` (cây rỗng) nếu không nút nào nằm trong đoạn. Ví dụ 3 nhấn mạnh điều này.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([1, 0, 2], 1, 2, [1, None, 2]),
    ([3, 0, 4, None, 2, None, None, 1], 1, 3, [3, 2, None, 1]),
    ([5], 1, 2, []),
]
for vals, low, high, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals      # dựng rồi xuất phải ra lại đúng list ban đầu
    print(f"root={vals}, low={low}, high={high} -> kỳ vọng {expected}")

root=[1, 0, 2], low=1, high=2 -> kỳ vọng [1, None, 2]
root=[3, 0, 4, None, 2, None, None, 1], low=1, high=3 -> kỳ vọng [3, 2, None, 1]
root=[5], low=1, high=2 -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. `TreeNode`, dựng cây và xuất cây

Mỗi nút của cây nhị phân có `val`, `left`, `right` (con trái, con phải; `None` là "không có"). Trên LeetCode cây được cho dưới dạng list **theo từng tầng, từ trái sang phải**, `None` đánh dấu chỗ trống. Hai hàm ở cell trên:

- `build_tree(values)`: dùng hàng đợi `deque`: lấy nút đang chờ ra, gán con trái rồi con phải từ list, rồi đưa các con thật vào hàng đợi.
- `tree_to_list(root)`: làm ngược lại và cắt các `None` thừa ở cuối. Ta dùng nó để **kiểm tra kết quả**.

Hãy dựng cây ví dụ 2 và nhìn các con của nó.

In [2]:
t = build_tree(examples[1][0])
print("gốc:", t.val, "| con trái:", t.left.val, "| con phải:", t.right.val)
print("con phải của 0:", t.left.right.val, "| con trái của 2:", t.left.right.left.val)
print("nút 4 có con?", t.right.left, t.right.right)
print(tree_to_list(t))
print("cây rỗng:", build_tree([]), tree_to_list(None))

gốc: 3 | con trái: 0 | con phải: 4
con phải của 0: 2 | con trái của 2: 1
nút 4 có con? None None
[3, 0, 4, None, 2, None, None, 1]
cây rỗng: None []


### 3.2. BST và duyệt trung thứ tự / tiền thứ tự

BST có tính chất: ở **mọi** nút, mọi giá trị ở cây con trái **nhỏ hơn** giá trị nút, mọi giá trị ở cây con phải **lớn hơn**. Hệ quả: duyệt **trung thứ tự** (trái → gốc → phải) cho dãy giá trị **tăng dần**. Duyệt **tiền thứ tự** (gốc → trái → phải) liệt kê gốc trước các con; ta sẽ dùng nó vì nó giữ thông tin về cấu trúc của BST (có thể dựng lại cây từ danh sách tiền thứ tự).

Để nhìn cây cho trực quan, hàm `draw` in cây **xoay ngang** (cây con phải ở trên, cây con trái ở dưới, thụt vào theo độ sâu).

In [3]:
def preorder(root):
    return [] if root is None else [root.val] + preorder(root.left) + preorder(root.right)

def inorder(root):
    return [] if root is None else inorder(root.left) + [root.val] + inorder(root.right)

def draw(root, depth=0):
    if root is None:
        return
    draw(root.right, depth + 1)
    print("    " * depth + str(root.val))
    draw(root.left, depth + 1)

t = build_tree([3, 0, 4, None, 2, None, None, 1])
print("preorder:", preorder(t))
print("inorder :", inorder(t), "(tăng dần vì là BST)")
print("cây (xoay ngang, phải ở trên):")
draw(t)

preorder: [3, 0, 2, 1, 4]
inorder : [0, 1, 2, 3, 4] (tăng dần vì là BST)
cây (xoay ngang, phải ở trên):
    4
3
        2
            1
    0


### 3.3. Đệ quy "trả về gốc của cây con đã xử lý"

Với cây, đệ quy thường có dạng: hàm `f(node)` **trả về gốc của cây con sau khi xử lý**. Cha nhận kết quả đó và **gán lại** vào con của mình: `node.left = f(node.left)`. Nhờ vậy một nút bị loại có thể **được thay** bởi một nút khác (ví dụ con của nó) mà cha không cần biết chi tiết. Đây chính là cách ta "nâng" cây con lên khi bỏ một nút.

Ví dụ nhỏ: hàm `mirror` lật cây bằng đúng khuôn mẫu này.

In [4]:
def mirror(node):
    if node is None:
        return None
    node.left, node.right = mirror(node.right), mirror(node.left)   # gán lại kết quả cho con
    return node

t = build_tree([1, 2, 3, 4])
print(tree_to_list(t), "->", tree_to_list(mirror(t)))

[1, 2, 3, 4] -> [1, 3, 2, None, None, None, 4]


### 3.4. Sửa con trỏ tại chỗ: `node.left = node.left.right`

Muốn "bỏ" một nút `x` là con trái của `p` và nâng con của `x` lên thay, ta chỉ đổi **một con trỏ**: `p.left = x.right` (hoặc `x.left`). Không cần xóa gì cả: nút `x` không còn ai trỏ tới nên bị Python thu dọn. Đây là thao tác chính của lời giải lặp.

In [5]:
t = build_tree([5, 0, 8, None, 3])           # 0 là con trái của 5, 3 là con phải của 0
print("trước:", tree_to_list(t))
t.left = t.left.right                        # bỏ nút 0, nâng 3 lên làm con trái của 5
print("sau  :", tree_to_list(t))

trước: [5, 0, 8, None, 3]
sau  : [5, 3, 8]


### 3.5. Ngăn xếp bằng `list` và duyệt tiền thứ tự **không đệ quy**

`list` làm ngăn xếp: `stack.append(x)` đẩy vào, `stack.pop()` lấy phần tử **mới nhất**. Duyệt tiền thứ tự bằng ngăn xếp: lấy nút ra, ghi giá trị, đẩy **con phải rồi con trái** (để con trái được lấy ra trước). Cách này tránh `RecursionError` với cây lệch rất sâu.

In [6]:
def preorder_iter(root):
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        out.append(node.val)
        stack.append(node.right)
        stack.append(node.left)
    return out

t = build_tree([3, 0, 4, None, 2, None, None, 1])
print(preorder_iter(t), preorder_iter(t) == preorder(t))

[3, 0, 2, 1, 4] True


### 3.6. Dựng BST bằng cách chèn, và cái bẫy "cùng giá trị, khác hình dạng"

`bst_insert(root, v)` chèn `v`: so với gốc, nhỏ hơn thì đi trái, lớn hơn thì đi phải, tới chỗ trống thì đặt nút mới. **Cùng một tập giá trị nhưng chèn theo thứ tự khác nhau cho hình dạng cây khác nhau.** Đó là lý do đề nhấn mạnh "giữ nguyên cấu trúc tương đối": đáp án là **một cây cụ thể**, không phải "bất kỳ BST nào chứa các giá trị đó".

In [7]:
def bst_insert(root, v):
    node = TreeNode(v)
    if root is None:
        return node
    cur = root
    while True:
        if v < cur.val:
            if cur.left is None:
                cur.left = node
                break
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = node
                break
            cur = cur.right
    return root

def build_bst(values):
    root = None
    for v in values:
        root = bst_insert(root, v)
    return root

a = build_bst([2, 1, 3])
b = build_bst([1, 2, 3])
c = build_bst([3, 2, 1])
print("chèn 2,1,3 ->", tree_to_list(a))
print("chèn 1,2,3 ->", tree_to_list(b), "(lệch phải)")
print("chèn 3,2,1 ->", tree_to_list(c), "(lệch trái)")
print("cùng tập giá trị:", inorder(a) == inorder(b) == inorder(c))

chèn 2,1,3 -> [2, 1, 3]
chèn 1,2,3 -> [1, None, 2, None, 3] (lệch phải)
chèn 3,2,1 -> [3, 2, None, 1] (lệch trái)
cùng tập giá trị: True


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `root = [3,0,4,null,2,null,null,1]`, `low = 1`, `high = 3`. Cây:

```
        3
      /   \
     0     4
      \
       2
      /
     1
```

Xét từng nút (giữ nếu giá trị trong `[1, 3]`):

| nút | giá trị | trong `[1,3]`? | số phận |
|---|---|---|---|
| gốc `3` | 3 | có | giữ |
| `0` | 0 | không (`0 < 1`) | bỏ |
| `4` | 4 | không (`4 > 3`) | bỏ |
| `2` | 2 | có | giữ |
| `1` | 1 | có | giữ |

Nút `0` bị bỏ nhưng **con phải của nó (`2`) vẫn phải còn**, và `2` vốn là hậu duệ của `3`, nên sau khi bỏ `0`, `2` phải được nâng lên làm con trái của `3`. Nút `2` giữ lại con trái `1`. Kết quả: `3 -> trái 2 -> trái 1`, tức `[3,2,null,1]`.

Hai quan sát then chốt về BST:

1. Nếu một nút `x` có `x.val < low` thì **mọi nút ở cây con trái của `x` đều nhỏ hơn `x.val`**, tức cũng `< low`: cả cây con trái **chắc chắn bị bỏ**. Chỉ còn cây con phải là có thể còn nút hợp lệ.
2. Tương tự nếu `x.val > high` thì cả cây con phải bị bỏ.

Hãy kiểm chứng bằng code: đếm xem trong cây ví dụ 2 những nút nào nằm trong cây con "bị bỏ chắc chắn".

In [8]:
def collect(root):
    return [] if root is None else collect(root.left) + [root.val] + collect(root.right)

t = build_tree(examples[1][0])
low, high = 1, 3
for node, name in [(t.left, "nút 0"), (t.right, "nút 4")]:
    vals = collect(node)
    print(f"{name}: cây con trái={collect(node.left)}, cây con phải={collect(node.right)}")
print("nút 0 < low  -> cây con TRÁI của nó (rỗng ở đây) chắc chắn bị bỏ; chỉ cây phải [2, 1] cần xét")
print("nút 4 > high -> cây con PHẢI của nó (rỗng ở đây) chắc chắn bị bỏ; chỉ cây trái cần xét")
print("các giá trị trong cây:", sorted(collect(t)), "| giữ lại:", [v for v in sorted(collect(t)) if low <= v <= high])

nút 0: cây con trái=[], cây con phải=[1, 2]
nút 4: cây con trái=[], cây con phải=[]
nút 0 < low  -> cây con TRÁI của nó (rỗng ở đây) chắc chắn bị bỏ; chỉ cây phải [2, 1] cần xét
nút 4 > high -> cây con PHẢI của nó (rỗng ở đây) chắc chắn bị bỏ; chỉ cây trái cần xét
các giá trị trong cây: [0, 1, 2, 3, 4] | giữ lại: [1, 2, 3]


Từ đây có ngay một ý tưởng: **dùng tính chất BST để bỏ cả cây con trong một bước**, thay vì xét từng nút một. Trước khi tối ưu, hãy làm bản "không cần nghĩ".

## 5. Cách 1 — Brute Force: thu các giá trị hợp lệ rồi chèn lại

### Ý tưởng

Không dùng tính chất BST để cắt nhánh:

1. Duyệt **toàn bộ** cây theo **tiền thứ tự** (gốc → trái → phải), thu giá trị vào list.
2. Lọc giữ các giá trị trong `[low, high]`.
3. Chèn lần lượt các giá trị đó (theo đúng thứ tự tiền thứ tự) vào một BST mới bằng `bst_insert`.

Vì sao ra đúng cây? Danh sách tiền thứ tự của cây đã cắt tỉa **chính là** danh sách tiền thứ tự của cây gốc sau khi lọc (cắt tỉa không đổi thứ tự tương đối của các nút còn lại). Và chèn các nút theo thứ tự tiền thứ tự của một BST sẽ **dựng lại đúng BST đó** (cha luôn được chèn trước con).

### Code

In [9]:
def trim_brute(root, low, high):
    vals = []
    stack = [root]
    while stack:                                   # tiền thứ tự, KHÔNG cắt nhánh
        node = stack.pop()
        if node is None:
            continue
        vals.append(node.val)
        stack.append(node.right)
        stack.append(node.left)
    new_root = None
    for v in vals:
        if low <= v <= high:
            new_root = bst_insert(new_root, v)
    return new_root

for vals, low, high, expected in examples:
    got = tree_to_list(trim_brute(build_tree(vals), low, high))
    print(vals, low, high, "->", got)
    assert got == expected

[1, 0, 2] 1 2 -> [1, None, 2]
[3, 0, 4, None, 2, None, None, 1] 1 3 -> [3, 2, None, 1]
[5] 1 2 -> []


### Độ phức tạp và nút thắt

- **Time:** duyệt `O(n)`, rồi mỗi lần chèn tốn `O(h)` (`h` là chiều cao cây mới), tổng `O(n · h)`; cây lệch có `h = n` nên xấu nhất `O(n²)`.
- **Space:** `O(n)` cho list giá trị và cây mới.

Nút thắt: (1) ta **xét mọi nút** dù BST cho phép bỏ cả cây con; (2) việc chèn lại **tạo nút mới** và tốn `O(h)` mỗi lần. Hãy đo trên cây lệch (chuỗi) để thấy `O(n²)`.

In [10]:
def timeit_once(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0

def chain_values(n):
    return list(range(1, n + 1))                 # chèn tăng dần -> cây lệch phải (chuỗi)

for n in (500, 1000, 2000):
    root = build_bst(chain_values(n))
    _, dt = timeit_once(trim_brute, root, 1, n)  # giữ hết: phải chèn lại n nút vào cây lệch
    print(f"n = {n:5d} (cây lệch) -> {dt * 1000:8.1f} ms")
print("thời gian tăng gấp ~4 lần khi n gấp đôi: đặc trưng O(n^2)")

n =   500 (cây lệch) ->      6.0 ms
n =  1000 (cây lệch) ->     19.2 ms


n =  2000 (cây lệch) ->     66.6 ms
thời gian tăng gấp ~4 lần khi n gấp đôi: đặc trưng O(n^2)


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** với BST, mỗi lần nhìn một nút ta biết **cả cây con nào chắc chắn bị bỏ**:

- `root.val > high` ⟹ `root` và **toàn bộ cây con phải** (toàn giá trị `> root.val > high`) bị bỏ. Đáp án nằm hoàn toàn trong cây con **trái**: hãy cắt tỉa cây con trái rồi **trả về nó** thay cho `root`.
- `root.val < low` ⟹ `root` và **toàn bộ cây con trái** bị bỏ. Trả về kết quả cắt tỉa cây con **phải**.
- Ngược lại `low <= root.val <= high`: `root` **được giữ**; cắt tỉa tiếp cả hai cây con: `root.left = trim(root.left)`, `root.right = trim(root.right)`; trả về `root`.

Đây là một đệ quy rất ngắn gồm **ba trường hợp**. Cũng nhờ nó mà ta không cần chạm vào những cây con bị bỏ (không duyệt chúng).

Kiểm chứng ý tưởng bằng số: xét ví dụ 2 và xem hàm sẽ làm gì tại mỗi nút (dùng bảng mục 4): tại `3` giữ → xuống `0` (`< low`) → đi sang con phải `2` → giữ, xuống `1` giữ → trả `2`; tại `4` (`> high`) → đi sang con trái (rỗng) → `None`.

In [11]:
def decision(val, low, high):
    if val > high:
        return "bỏ nút và cây phải -> xét cây TRÁI"
    if val < low:
        return "bỏ nút và cây trái -> xét cây PHẢI"
    return "giữ nút -> cắt tỉa cả hai con"

for v in (3, 0, 4, 2, 1):
    print(f"nút {v} với [1,3]: {decision(v, 1, 3)}")

nút 3 với [1,3]: giữ nút -> cắt tỉa cả hai con
nút 0 với [1,3]: bỏ nút và cây trái -> xét cây PHẢI
nút 4 với [1,3]: bỏ nút và cây phải -> xét cây TRÁI
nút 2 với [1,3]: giữ nút -> cắt tỉa cả hai con
nút 1 với [1,3]: giữ nút -> cắt tỉa cả hai con


## 7. Các cách tiếp theo

Tổng cộng 4 cách (đều khác nhau thật về ý tưởng):

1. Thu giá trị hợp lệ rồi chèn lại (đã làm, `O(n·h)`).
2. **Đệ quy 3 trường hợp** (lời giải nộp).
3. **Lặp, sửa con trỏ tại chỗ**: đưa gốc vào đoạn, rồi tỉa xương sống trái và xương sống phải.
4. **Tiền thứ tự có cắt nhánh + dựng lại cây `O(n)` bằng ngăn xếp**, không sửa cây gốc.

### 7.1. Cách 2 — Đệ quy ba trường hợp

Giải thích từng đoạn:

1. `if root is None: return None`: cây rỗng thì kết quả là rỗng.
2. `if root.val > high: return trim(root.left)`: gốc quá lớn, bỏ nó **cùng cây phải**, nghiệm là cắt tỉa của cây trái.
3. `if root.val < low: return trim(root.right)`: ngược lại.
4. Ngược lại giữ gốc: `root.left = trim(root.left)`, `root.right = trim(root.right)`, rồi `return root`.

Chú ý **phải gán lại** kết quả cho `root.left` và `root.right` (mục 3.3); nếu chỉ gọi `trim(root.left)` mà không gán thì cây không đổi.

In [12]:
def trim_recursive(root, low, high):
    if root is None:
        return None
    if root.val > high:
        return trim_recursive(root.left, low, high)
    if root.val < low:
        return trim_recursive(root.right, low, high)
    root.left = trim_recursive(root.left, low, high)
    root.right = trim_recursive(root.right, low, high)
    return root

for vals, low, high, expected in examples:
    got = tree_to_list(trim_recursive(build_tree(vals), low, high))
    print(vals, low, high, "->", got)
    assert got == expected

[1, 0, 2] 1 2 -> [1, None, 2]
[3, 0, 4, None, 2, None, None, 1] 1 3 -> [3, 2, None, 1]
[5] 1 2 -> []


In [13]:
class Solution2:
    def trimBST(self, root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
        def dfs(node):
            if node is None:
                return None
            if node.val > high:
                return dfs(node.left)
            if node.val < low:
                return dfs(node.right)
            node.left = dfs(node.left)
            node.right = dfs(node.right)
            return node

        return dfs(root)

for vals, low, high, expected in examples:
    assert tree_to_list(Solution2().trimBST(build_tree(vals), low, high)) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp Cách 2:** Time `O(n)` (mỗi nút được thăm tối đa một lần; nhiều cây con bị bỏ qua hoàn toàn); Space `O(h)` cho ngăn xếp đệ quy, `h` là chiều cao cây, tối đa `O(n)` với cây lệch.

### 7.2. Cách 3 — Lặp, sửa con trỏ tại chỗ (xương sống trái/phải)

Ý tưởng: làm thành **ba pha** không đệ quy.

**Pha 1 — đưa gốc vào đoạn.** Chừng nào `root` còn nằm ngoài `[low, high]`: nếu `root.val < low` thì `root = root.right`, nếu `root.val > high` thì `root = root.left`. Nếu hết cây (`root` thành `None`) thì trả `None`.

Sau pha 1, `root` nằm trong đoạn. Ta chứng minh được: **mọi nút của cây con trái của `root` đều `< root.val <= high`**, nên cây con trái chỉ có thể vi phạm biên **`low`**. Tương tự cây con phải chỉ có thể vi phạm biên **`high`**.

**Pha 2 — tỉa xương sống trái (chỉ lo `low`).** Đặt `node = root`. Lặp: chừng nào `node.left` tồn tại và `node.left.val < low`, thay `node.left = node.left.right` (bỏ con trái và cây con trái của nó, nâng cây con phải lên). Khi `node.left` đã trong đoạn (hoặc rỗng), đi xuống `node = node.left` và lặp. Dừng khi `node` rỗng.

**Pha 3 — tỉa xương sống phải (chỉ lo `high`).** Đối xứng: `while node.right and node.right.val > high: node.right = node.right.left`, rồi `node = node.right`.

Vì sao pha 2 chỉ cần đi dọc **xương sống trái**? Khi `node.left.val >= low` ta giữ `node.left`, và cây con **phải** của `node.left` có giá trị `> node.left.val >= low` nên **không** vi phạm `low`; chỉ cây con **trái** còn có thể vi phạm, nên đi tiếp xuống trái.

In [14]:
def trim_iterative(root, low, high):
    # Pha 1: đưa gốc vào đoạn [low, high]
    while root and (root.val < low or root.val > high):
        root = root.right if root.val < low else root.left
    if root is None:
        return None
    # Pha 2: tỉa xương sống trái (chỉ có thể vi phạm low)
    node = root
    while node:
        while node.left and node.left.val < low:
            node.left = node.left.right
        node = node.left
    # Pha 3: tỉa xương sống phải (chỉ có thể vi phạm high)
    node = root
    while node:
        while node.right and node.right.val > high:
            node.right = node.right.left
        node = node.right
    return root

for vals, low, high, expected in examples:
    got = tree_to_list(trim_iterative(build_tree(vals), low, high))
    print(vals, low, high, "->", got)
    assert got == expected

[1, 0, 2] 1 2 -> [1, None, 2]
[3, 0, 4, None, 2, None, None, 1] 1 3 -> [3, 2, None, 1]
[5] 1 2 -> []


In [15]:
class Solution3:
    def trimBST(self, root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
        while root and (root.val < low or root.val > high):
            root = root.right if root.val < low else root.left
        if root is None:
            return None
        node = root
        while node:
            while node.left and node.left.val < low:
                node.left = node.left.right
            node = node.left
        node = root
        while node:
            while node.right and node.right.val > high:
                node.right = node.right.left
            node = node.right
        return root

for vals, low, high, expected in examples:
    assert tree_to_list(Solution3().trimBST(build_tree(vals), low, high)) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp Cách 3:** Time `O(n)` hay chính xác hơn `O(độ dài ba đường đi)`: mỗi con trỏ bị bỏ qua chỉ một lần, tối đa `O(h)` bước trên các xương sống; chắc chắn `O(n)`. Space `O(1)` phụ (không đệ quy). Cây gốc bị **sửa tại chỗ**.

### 7.3. Cách 4 — Tiền thứ tự có cắt nhánh, rồi dựng lại cây bằng ngăn xếp

Ý tưởng kết hợp hai điều đã biết:

1. Duyệt **tiền thứ tự bằng ngăn xếp** nhưng **cắt nhánh** bằng tính chất BST: nút `< low` thì chỉ đẩy con phải; nút `> high` thì chỉ đẩy con trái; nút trong đoạn thì ghi giá trị và đẩy cả hai con. Kết quả là danh sách tiền thứ tự của cây đã cắt tỉa (giữ nguyên thứ tự tương đối).
2. **Dựng lại BST từ danh sách tiền thứ tự trong `O(n)`** bằng một ngăn xếp: giữ ngăn xếp các nút "còn có thể nhận con phải" (đường đi từ gốc xuống nút mới nhất). Với giá trị `v` mới: nếu `v < ngăn_xếp[-1].val` thì là con **trái** của đỉnh. Ngược lại, lấy ra khỏi ngăn xếp mọi nút có giá trị `< v` (nút cuối cùng bị lấy ra `last` là cha) và đặt `v` làm con **phải** của `last`. Rồi đẩy nút mới vào ngăn xếp.

Cách này **không sửa cây gốc** (tạo cây mới), rất hợp khi bạn cần giữ nguyên đầu vào.

In [16]:
def preorder_pruned(root, low, high):
    vals, stack = [], [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if node.val < low:
            stack.append(node.right)          # cả cây trái bị bỏ
        elif node.val > high:
            stack.append(node.left)           # cả cây phải bị bỏ
        else:
            vals.append(node.val)
            stack.append(node.right)
            stack.append(node.left)           # đẩy trái sau để lấy ra trước
    return vals

def build_from_preorder(vals):
    if not vals:
        return None
    root = TreeNode(vals[0])
    stack = [root]
    for v in vals[1:]:
        node = TreeNode(v)
        if v < stack[-1].val:
            stack[-1].left = node
        else:
            last = None
            while stack and stack[-1].val < v:
                last = stack.pop()
            last.right = node
        stack.append(node)
    return root

def trim_preorder_rebuild(root, low, high):
    return build_from_preorder(preorder_pruned(root, low, high))

# thử trước hai hàm con
t = build_tree(examples[1][0])
pv = preorder_pruned(t, 1, 3)
print("tiền thứ tự đã cắt nhánh:", pv)
print("dựng lại:", tree_to_list(build_from_preorder(pv)))
for vals, low, high, expected in examples:
    got = tree_to_list(trim_preorder_rebuild(build_tree(vals), low, high))
    print(vals, low, high, "->", got)
    assert got == expected

tiền thứ tự đã cắt nhánh: [3, 2, 1]
dựng lại: [3, 2, None, 1]
[1, 0, 2] 1 2 -> [1, None, 2]
[3, 0, 4, None, 2, None, None, 1] 1 3 -> [3, 2, None, 1]
[5] 1 2 -> []


In [17]:
class Solution4:
    def trimBST(self, root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
        vals, stack = [], [root]
        while stack:
            node = stack.pop()
            if node is None:
                continue
            if node.val < low:
                stack.append(node.right)
            elif node.val > high:
                stack.append(node.left)
            else:
                vals.append(node.val)
                stack.append(node.right)
                stack.append(node.left)
        if not vals:
            return None
        new_root = TreeNode(vals[0])
        stack = [new_root]
        for v in vals[1:]:
            node = TreeNode(v)
            if v < stack[-1].val:
                stack[-1].left = node
            else:
                last = None
                while stack and stack[-1].val < v:
                    last = stack.pop()
                last.right = node
            stack.append(node)
        return new_root

for vals, low, high, expected in examples:
    assert tree_to_list(Solution4().trimBST(build_tree(vals), low, high)) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp Cách 4:** Time `O(n)` (mỗi nút được đẩy vào ngăn xếp tối đa một lần ở pha duyệt; pha dựng cũng mỗi nút vào/ra ngăn xếp một lần); Space `O(n)` (list giá trị, ngăn xếp, cây mới). **Không** sửa cây gốc.

Bảng so sánh các cách ở mục 16.

## 8. TRACE từng bước

Trace Cách 2 (đệ quy) trên ví dụ 2: `root = [3,0,4,null,2,null,null,1]`, `low = 1`, `high = 3`.

| bước | gọi `dfs(node)` | so sánh | hành động | trả về |
|---|---|---|---|---|
| 1 | `dfs(3)` | `1 <= 3 <= 3` | giữ; gọi `dfs(3.left)` rồi `dfs(3.right)` | (chờ) |
| 2 | `dfs(0)` | `0 < 1` | bỏ `0` và cây trái; trả `dfs(0.right)` | (chờ) |
| 3 | `dfs(2)` | `1 <= 2 <= 3` | giữ; gọi `dfs(2.left)`, `dfs(2.right)` | (chờ) |
| 4 | `dfs(1)` | `1 <= 1 <= 3` | giữ; hai con rỗng | nút `1` |
| 5 | `dfs(None)` (con phải của 2) | rỗng | | `None` |
| 6 | (xong `dfs(2)`) | | `2.left = 1`, `2.right = None` | nút `2` |
| 7 | (xong `dfs(0)`) | | | nút `2` |
| 8 | `3.left = nút 2` | | | |
| 9 | `dfs(4)` | `4 > 3` | bỏ `4` và cây phải; trả `dfs(4.left) = dfs(None)` | `None` |
| 10 | `3.right = None`; xong `dfs(3)` | | | nút `3` |

Cây kết quả: `3 -> trái 2 -> trái 1`, tức `[3,2,null,1]`. Hàm dưới in từng lời gọi với thụt lề theo độ sâu đệ quy.

In [18]:
def trace_recursive(root, low, high):
    def dfs(node, depth):
        pad = "  " * depth
        if node is None:
            print(f"{pad}dfs(None) -> None")
            return None
        if node.val > high:
            print(f"{pad}dfs({node.val}): {node.val} > {high} -> bỏ nút và cây phải, đi sang TRÁI")
            res = dfs(node.left, depth + 1)
        elif node.val < low:
            print(f"{pad}dfs({node.val}): {node.val} < {low} -> bỏ nút và cây trái, đi sang PHẢI")
            res = dfs(node.right, depth + 1)
        else:
            print(f"{pad}dfs({node.val}): trong [{low}, {high}] -> giữ, xử lý hai con")
            node.left = dfs(node.left, depth + 1)
            node.right = dfs(node.right, depth + 1)
            res = node
        print(f"{pad}=> trả về {res.val if res else None}")
        return res
    out = dfs(root, 0)
    print("kết quả:", tree_to_list(out))
    return out

assert tree_to_list(trace_recursive(build_tree([3, 0, 4, None, 2, None, None, 1]), 1, 3)) == [3, 2, None, 1]

dfs(3): trong [1, 3] -> giữ, xử lý hai con
  dfs(0): 0 < 1 -> bỏ nút và cây trái, đi sang PHẢI
    dfs(2): trong [1, 3] -> giữ, xử lý hai con
      dfs(1): trong [1, 3] -> giữ, xử lý hai con
        dfs(None) -> None
        dfs(None) -> None
      => trả về 1
      dfs(None) -> None
    => trả về 2
  => trả về 2
  dfs(4): 4 > 3 -> bỏ nút và cây phải, đi sang TRÁI
    dfs(None) -> None
  => trả về None
=> trả về 3
kết quả: [3, 2, None, 1]


Trace Cách 3 (lặp) trên một cây có **gốc nằm ngoài đoạn**: `root = [5,3,8,1,4,7,9]`, `low = 4`, `high = 8`. Gốc `5` nằm trong đoạn nên pha 1 không làm gì; pha 2 bỏ `3` (vì `3 < 4`) và nâng `4` lên; pha 3 bỏ `9` (vì `9 > 8`) và nâng con trái của `9` (rỗng) lên.

In [19]:
def trace_iterative(root, low, high):
    print(f"low={low}, high={high}, cây ban đầu: {tree_to_list(root)}")
    # Pha 1
    while root and (root.val < low or root.val > high):
        nxt = root.right if root.val < low else root.left
        print(f"pha 1: gốc {root.val} ngoài đoạn -> đi sang {'phải' if root.val < low else 'trái'} ({nxt.val if nxt else None})")
        root = nxt
    if root is None:
        print("pha 1: hết cây -> None")
        return None
    print(f"pha 1 xong: gốc mới = {root.val}")
    # Pha 2
    node = root
    while node:
        while node.left and node.left.val < low:
            print(f"pha 2: tại {node.val}, con trái {node.left.val} < {low} -> node.left = node.left.right")
            node.left = node.left.right
        node = node.left
    print("sau pha 2:", tree_to_list(root))
    # Pha 3
    node = root
    while node:
        while node.right and node.right.val > high:
            print(f"pha 3: tại {node.val}, con phải {node.right.val} > {high} -> node.right = node.right.left")
            node.right = node.right.left
        node = node.right
    print("sau pha 3:", tree_to_list(root))
    return root

assert tree_to_list(trace_iterative(build_tree([5, 3, 8, 1, 4, 7, 9]), 4, 8)) == [5, 4, 8, None, None, 7]

low=4, high=8, cây ban đầu: [5, 3, 8, 1, 4, 7, 9]
pha 1 xong: gốc mới = 5
pha 2: tại 5, con trái 3 < 4 -> node.left = node.left.right
sau pha 2: [5, 4, 8, None, None, 7, 9]
pha 3: tại 8, con phải 9 > 8 -> node.right = node.right.left
sau pha 3: [5, 4, 8, None, None, 7]


Một trace nữa với gốc **bị loại** (pha 1 có việc làm): `root = [3,0,4,null,2,null,null,1]`, `low = 2`, `high = 3`. Gốc `3` trong đoạn; thử `low = 4`, `high = 9` cho gốc `3 < 4` phải đi sang phải.

In [20]:
assert tree_to_list(trace_iterative(build_tree([3, 0, 4, None, 2, None, None, 1]), 4, 9)) == [4]
print()
assert tree_to_list(trace_iterative(build_tree([5, 2, 9, 1, 4, 7, 10, None, None, 3]), 3, 7)) == [5, 4, 7, 3]

low=4, high=9, cây ban đầu: [3, 0, 4, None, 2, None, None, 1]
pha 1: gốc 3 ngoài đoạn -> đi sang phải (4)
pha 1 xong: gốc mới = 4
sau pha 2: [4]
sau pha 3: [4]

low=3, high=7, cây ban đầu: [5, 2, 9, 1, 4, 7, 10, None, None, 3]
pha 1 xong: gốc mới = 5
pha 2: tại 5, con trái 2 < 3 -> node.left = node.left.right
sau pha 2: [5, 4, 9, 3, None, 7, 10]
pha 3: tại 5, con phải 9 > 7 -> node.right = node.right.left
sau pha 3: [5, 4, 7, 3]


## 9. Code nộp LeetCode

Ta chọn **Cách 2 (đệ quy ba trường hợp)**: ngắn, rõ, `O(n)` và là lời giải chuẩn (có thể thay bằng Cách 3 nếu lo giới hạn độ sâu đệ quy).

In [21]:
class Solution:
    def trimBST(self, root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
        def dfs(node):
            if node is None:
                return None
            if node.val > high:
                return dfs(node.left)
            if node.val < low:
                return dfs(node.right)
            node.left = dfs(node.left)
            node.right = dfs(node.right)
            return node

        return dfs(root)


sol = Solution()
for vals, low, high, expected in examples:
    got = tree_to_list(sol.trimBST(build_tree(vals), low, high))
    print(vals, low, high, "->", got)
    assert got == expected

[1, 0, 2] 1 2 -> [1, None, 2]
[3, 0, 4, None, 2, None, None, 1] 1 3 -> [3, 2, None, 1]
[5] 1 2 -> []


## 10. Vì sao đúng

**Khẳng định 1 (ba trường hợp).** Gọi `T(x)` là cây đáp án khi cắt tỉa cây con gốc `x`.

- *`x = None`:* hiển nhiên `T = None`.
- *`x.val > high`:* mọi nút ở cây con phải của `x` có giá trị `> x.val > high`, nên **không nút nào** của `x` và cây con phải được giữ. Mọi nút giữ lại đều nằm trong cây con trái; ta cần `T(x.left)`. Gốc của đáp án chính là gốc của `T(x.left)`, và cấu trúc tương đối được giữ vì các nút giữ lại vốn đã là hậu duệ của nhau theo đúng cây con trái.
- *`x.val < low`:* đối xứng, đáp án là `T(x.right)`.
- *`low <= x.val <= high`:* `x` được giữ và là tổ tiên của mọi nút còn lại trong cây con của nó, nên phải là gốc của đáp án. Cây con trái của đáp án gồm các nút giữ lại của cây con trái (đều `< x.val`) và cây con phải tương tự; theo quy nạp theo chiều cao, chúng là `T(x.left)` và `T(x.right)`. Gán `x.left = T(x.left)`, `x.right = T(x.right)`.

**Khẳng định 2 (kết quả đúng và duy nhất).** Tập nút giữ lại đúng là các nút có giá trị trong `[low, high]` (mỗi nút được xét đúng một lần, hoặc nằm trong cây con bị loại chắc chắn). Hậu duệ của mỗi nút được giữ vẫn là hậu duệ: nếu `u` là tổ tiên của `v` ở cây gốc và cả hai được giữ, thì ở mọi bước đệ quy từ `u` đi xuống `v` ta không loại `u` hoặc `v`, nên `v` vẫn nằm dưới `u` sau khi các nút ở giữa bị loại. Tính **duy nhất** (theo đề) cho phép ta dùng kiểm tra độc lập ở mục 12: "tập hậu duệ giữ nguyên" và "kết quả là BST nằm trong đoạn".

**Khẳng định 3 (Cách 3 đúng).** Pha 1 duyệt như Cách 2 cho tới khi gốc nằm trong đoạn (hai nhánh đầu của đệ quy). Sau đó, mọi nút trong cây con trái của `root` có giá trị `< root.val <= high`, nên chỉ có thể vi phạm `low`. Ở `node` với `node.left.val < low`, ta thay `node.left` bằng `node.left.right` lặp lại cho tới khi `node.left` hợp lệ: đây là cách bỏ `node.left` và cả cây con trái của nó (toàn `< low`). Khi `node.left.val >= low`, cây con phải của `node.left` toàn `> node.left.val >= low` nên không vi phạm; chỉ cây con trái cần xét tiếp, nên đi xuống `node.left`. Pha 3 đối xứng. Hai pha không phá nhau: pha 2 chỉ đổi các con trỏ `left` và chỉ gỡ nút `< low`; pha 3 chỉ đổi các con trỏ `right` và chỉ gỡ nút `> high`. Sau pha 2 không còn nút `< low`, và pha 3 gỡ nút `> high` không làm xuất hiện nút `< low` (vì nó chỉ nâng cây con trái của nút bị gỡ, mà cây này vẫn nằm trong cận dưới). Điều này cũng được kiểm chứng ngẫu nhiên ở mục 12.

**Khẳng định 4 (Cách 4 đúng).** Duyệt có cắt nhánh sinh đúng danh sách tiền thứ tự của cây đáp án (gốc trước, rồi cây con trái đã cắt tỉa, rồi cây con phải đã cắt tỉa); và BST được xác định duy nhất bởi tiền thứ tự của nó (thuật toán dựng bằng ngăn xếp đặt mỗi giá trị đúng vị trí).

## 11. Time / Space Complexity

- **Cách 1 (thu và chèn lại):** Time `O(n · h)` (xấu nhất `O(n²)`); Space `O(n)`.
- **Cách 2 (đệ quy, nộp):** Time `O(n)`; Space `O(h)` (ngăn xếp đệ quy; xấu nhất `O(n)`).
- **Cách 3 (lặp, sửa tại chỗ):** Time `O(n)`; Space `O(1)` phụ.
- **Cách 4 (tiền thứ tự + dựng lại):** Time `O(n)`; Space `O(n)`; không sửa cây gốc.

Lý do `O(n)` của cách nộp: mỗi lời gọi `dfs` xử lý đúng một nút và đệ quy xuống tối đa hai con; không nút nào được thăm hai lần. Hãy đo với `n = 10^4` trên hai dạng cây: cây BST ngẫu nhiên (thấp) và cây lệch (sâu `10^4`). Với cây lệch, đệ quy cần nâng giới hạn đệ quy.

In [22]:
def random_bst_list(rng, n, lo=0, hi=10 ** 4):
    """Cây BST ngẫu nhiên n nút, giá trị phân biệt trong [lo, hi], trả về list kiểu LeetCode."""
    vals = rng.sample(range(lo, hi + 1), n)
    return tree_to_list(build_bst(vals))

rng = random.Random(7)
N = 10 ** 4
cases = {
    "BST ngẫu nhiên (thấp)": (random_bst_list(rng, N), 2500, 7500),
    "cây lệch phải (sâu 10^4)": (tree_to_list(build_bst(range(0, N))), 2500, 7500),
}
old = sys.getrecursionlimit()
sys.setrecursionlimit(50000)
try:
    for name, (lst, low, high) in cases.items():
        outs = []
        for mname, fn in (
            ("đệ quy", lambda r: Solution2().trimBST(r, low, high)),
            ("lặp", lambda r: Solution3().trimBST(r, low, high)),
            ("preorder+dựng", lambda r: Solution4().trimBST(r, low, high)),
        ):
            root = build_tree(lst)
            res, dt = timeit_once(fn, root)
            outs.append((mname, tree_to_list(res), dt))
        assert outs[0][1] == outs[1][1] == outs[2][1]
        print(f"{name:26s} | " + " | ".join(f"{m}: {dt * 1000:6.1f} ms" for m, _, dt in outs))
finally:
    sys.setrecursionlimit(old)

BST ngẫu nhiên (thấp)      | đệ quy:    0.9 ms | lặp:    0.2 ms | preorder+dựng:    2.6 ms
cây lệch phải (sâu 10^4)   | đệ quy:    1.6 ms | lặp:    0.5 ms | preorder+dựng:    2.5 ms


Cả ba cách chạy trong vài chục mili-giây ở cỡ tối đa. Lưu ý: ở cây lệch sâu `10^4`, bản đệ quy chỉ chạy được nhờ đã nâng giới hạn đệ quy; mục 13 cho thấy điều gì xảy ra khi để mặc định.

## 12. Bộ test

Ta kiểm tra bốn lớp: ví dụ của đề, edge case, phản ví dụ cho các cách "gần đúng", và đối chiếu có hệ thống/ngẫu nhiên (có seed) với **hai kiểm tra độc lập**:

1. **Oracle thu-và-chèn-lại** (Cách 1).
2. **Kiểm tra tính chất** `validate_trim`: kết quả là BST hợp lệ, tập giá trị đúng bằng các giá trị gốc nằm trong `[low, high]`, và **tập cặp (tổ tiên, hậu duệ) giữ nguyên** (đúng như đề yêu cầu).

In [23]:
def is_bst(root, lo=float("-inf"), hi=float("inf")):
    if root is None:
        return True
    if not (lo < root.val < hi):
        return False
    return is_bst(root.left, lo, root.val) and is_bst(root.right, root.val, hi)

def ancestor_pairs(root):
    """Tập các cặp (tổ tiên, hậu duệ) theo giá trị (giá trị đã là duy nhất)."""
    pairs = set()
    def dfs(node, path):
        if node is None:
            return
        for a in path:
            pairs.add((a, node.val))
        path.append(node.val)
        dfs(node.left, path)
        dfs(node.right, path)
        path.pop()
    dfs(root, [])
    return pairs

def validate_trim(orig_list, low, high, result_root):
    orig = build_tree(orig_list)
    keep = {v for v in collect(orig) if low <= v <= high}
    assert is_bst(result_root), "kết quả không phải BST"
    assert set(collect(result_root)) == keep, "sai tập giá trị giữ lại"
    assert len(collect(result_root)) == len(keep), "có giá trị trùng"
    expected_pairs = {(a, b) for (a, b) in ancestor_pairs(orig) if a in keep and b in keep}
    assert ancestor_pairs(result_root) == expected_pairs, "cấu trúc tương đối bị thay đổi"
    return True

# kiểm tra chính hàm validate_trim trên ví dụ đề
for vals, low, high, expected in examples:
    validate_trim(vals, low, high, build_tree(expected))
print("validate_trim đồng ý với đáp án mẫu của đề")

validate_trim đồng ý với đáp án mẫu của đề


In [24]:
all_methods = {
    "brute": trim_brute,
    "recursive": trim_recursive,
    "iterative": trim_iterative,
    "preorder-rebuild": trim_preorder_rebuild,
    "Sol2": lambda r, lo, hi: Solution2().trimBST(r, lo, hi),
    "Sol3": lambda r, lo, hi: Solution3().trimBST(r, lo, hi),
    "Sol4": lambda r, lo, hi: Solution4().trimBST(r, lo, hi),
    "submit": lambda r, lo, hi: Solution().trimBST(r, lo, hi),
}

def run_all(lst, low, high, expected=None):
    exp = tree_to_list(trim_brute(build_tree(lst), low, high)) if expected is None else expected
    for name, fn in all_methods.items():
        res = fn(build_tree(lst), low, high)           # mỗi cách nhận một cây MỚI (vì có cách sửa tại chỗ)
        assert tree_to_list(res) == exp, (name, lst, low, high, tree_to_list(res), exp)
        validate_trim(lst, low, high, res)
    return exp

for vals, low, high, expected in examples:
    assert run_all(vals, low, high, expected) == expected
print("ví dụ đề: OK (mọi cách khớp đáp án mẫu và qua validate_trim)")

ví dụ đề: OK (mọi cách khớp đáp án mẫu và qua validate_trim)


In [25]:
# Edge case
edge = [
    ([5], 1, 2, []),                               # một nút bị cắt hết
    ([5], 5, 5, [5]),                              # một nút nằm đúng biên
    ([2, 1, 3], 2, 3, [2, None, 3]),               # bỏ con trái
    ([2, 1, 3], 1, 2, [2, 1]),                     # bỏ con phải
    ([2, 1, 3], 0, 10, [2, 1, 3]),                 # đoạn phủ hết: giữ nguyên
    ([2, 1, 3], 4, 9, []),                         # đoạn nằm hoàn toàn bên phải cây
    ([2, 1, 3], 0, 0, []),                         # đoạn nằm hoàn toàn bên trái cây
    ([5, 3, 8, 1, 4, 7, 9], 4, 8, [5, 4, 8, None, None, 7]),
    ([3, 0, 4, None, 2, None, None, 1], 4, 9, [4]),          # gốc bị loại, đi phải
    ([3, 0, 4, None, 2, None, None, 1], 0, 0, [0]),   # chỉ còn nút 0
    ([1, None, 2, None, 3, None, 4], 2, 3, [2, None, 3]),    # cây lệch phải
    ([4, 3, None, 2, None, 1], 2, 3, [3, 2]),                # cây lệch trái
]
for lst, low, high, expected in edge:
    got = run_all(lst, low, high)
    print(f"{lst}, [{low},{high}] -> {got}")
    assert got == expected, (lst, low, high, got, expected)
print("edge case: OK")

[5], [1,2] -> []
[5], [5,5] -> [5]
[2, 1, 3], [2,3] -> [2, None, 3]
[2, 1, 3], [1,2] -> [2, 1]
[2, 1, 3], [0,10] -> [2, 1, 3]
[2, 1, 3], [4,9] -> []
[2, 1, 3], [0,0] -> []
[5, 3, 8, 1, 4, 7, 9], [4,8] -> [5, 4, 8, None, None, 7]
[3, 0, 4, None, 2, None, None, 1], [4,9] -> [4]
[3, 0, 4, None, 2, None, None, 1], [0,0] -> [0]
[1, None, 2, None, 3, None, 4], [2,3] -> [2, None, 3]
[4, 3, None, 2, None, 1], [2,3] -> [3, 2]
edge case: OK


Phản ví dụ cho các cách "gần đúng" mà người mới hay nghĩ ra:

- *Chỉ lọc giá trị rồi dựng BST cân bằng từ dãy đã sắp xếp*: giữ đúng tập giá trị nhưng **làm mất cấu trúc tương đối** nên không phải đáp án (đề yêu cầu giữ nguyên).
- *Chỉ cắt các lá nằm ngoài đoạn*: bỏ sót các nút trong ngoài đoạn nhưng có con còn hợp lệ.
- *Đi nhầm hướng khi gốc ra ngoài đoạn* (gặp `root.val < low` lại trả `root.left`): sai.

In [26]:
def wrong_balanced_rebuild(root, low, high):
    vals = sorted(v for v in collect(root) if low <= v <= high)
    def build(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        return TreeNode(vals[mid], build(lo, mid - 1), build(mid + 1, hi))
    return build(0, len(vals) - 1)

def wrong_direction(root, low, high):
    if root is None:
        return None
    if root.val > high:
        return wrong_direction(root.right, low, high)      # BUG: đi nhầm sang phải
    if root.val < low:
        return wrong_direction(root.left, low, high)       # BUG: đi nhầm sang trái
    root.left = wrong_direction(root.left, low, high)
    root.right = wrong_direction(root.right, low, high)
    return root

lst, low, high = [3, 0, 4, None, 2, None, None, 1], 1, 3
good = tree_to_list(Solution().trimBST(build_tree(lst), low, high))
print("đúng           :", good)
bal = wrong_balanced_rebuild(build_tree(lst), low, high)
print("dựng cân bằng  :", tree_to_list(bal), "| đủ giá trị:", sorted(collect(bal)) == [1, 2, 3], "| giống đáp án:", tree_to_list(bal) == good)
print("đi nhầm hướng  :", tree_to_list(wrong_direction(build_tree(lst), low, high)))
assert tree_to_list(bal) != good
assert tree_to_list(wrong_direction(build_tree(lst), low, high)) != good

đúng           : [3, 2, None, 1]
dựng cân bằng  : [2, 1, 3] | đủ giá trị: True | giống đáp án: False
đi nhầm hướng  : [3]


In [27]:
# Đối chiếu có hệ thống: MỌI cây BST sinh từ MỌI thứ tự chèn của {1..5} (120 cây), MỌI cặp 0 <= low <= high <= 6
from itertools import permutations
total = 0
for order in permutations(range(1, 6)):
    lst = tree_to_list(build_bst(order))
    for low in range(0, 7):
        for high in range(low, 7):
            run_all(lst, low, high)
            total += 1
print("vét cạn", total, "ca (120 cây x 28 đoạn): mọi cách khớp oracle và qua validate_trim")

vét cạn 3360 ca (120 cây x 28 đoạn): mọi cách khớp oracle và qua validate_trim


In [28]:
# Ngẫu nhiên có seed, đúng ràng buộc (giá trị 0..10^4 duy nhất, n >= 1)
rng = random.Random(2027)
empty_results = 0
for t in range(600):
    n = rng.randint(1, 40)
    hi_val = rng.choice([n + 3, 60, 10 ** 4])
    lst = random_bst_list(rng, n, 0, hi_val)
    low = rng.randint(0, hi_val)
    high = rng.randint(low, hi_val)
    res = run_all(lst, low, high)
    empty_results += (res == [])
print("600 ca ngẫu nhiên: OK, trong đó", empty_results, "ca ra cây rỗng")

# Cây lớn: dùng các cách không đệ quy (và đệ quy với giới hạn đã nâng)
old = sys.getrecursionlimit()
sys.setrecursionlimit(50000)
try:
    rng = random.Random(5)
    for lst in (random_bst_list(rng, 10 ** 4), tree_to_list(build_bst(range(10 ** 4))), tree_to_list(build_bst(range(9999, -1, -1)))):
        for low, high in ((0, 10 ** 4), (3000, 3000), (2500, 7500), (9990, 10 ** 4), (0, 5)):
            base = tree_to_list(Solution4().trimBST(build_tree(lst), low, high))
            for name in ("recursive", "iterative", "Sol2", "Sol3", "submit"):
                assert tree_to_list(all_methods[name](build_tree(lst), low, high)) == base, name
finally:
    sys.setrecursionlimit(old)
print("test n = 10^4 (cây ngẫu nhiên, lệch phải, lệch trái): OK")

600 ca ngẫu nhiên: OK, trong đó 84 ca ra cây rỗng


test n = 10^4 (cây ngẫu nhiên, lệch phải, lệch trái): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gọi đệ quy nhưng không gán lại kết quả

Code sai:

```python
if low <= root.val <= high:
    trim(root.left)      # BUG: kết quả bị bỏ đi
    trim(root.right)
    return root
```

Hàm trả về **gốc mới** của cây con (có thể là một nút khác `root.left`). Nếu không gán lại `root.left = trim(root.left)`, cây không đổi.

In [29]:
def buggy_no_assign(root, low, high):
    if root is None:
        return None
    if root.val > high:
        return buggy_no_assign(root.left, low, high)
    if root.val < low:
        return buggy_no_assign(root.right, low, high)
    buggy_no_assign(root.left, low, high)          # BUG: không gán lại
    buggy_no_assign(root.right, low, high)
    return root

lst = [3, 0, 4, None, 2, None, None, 1]
print("sai  :", tree_to_list(buggy_no_assign(build_tree(lst), 1, 3)), "(vẫn còn 0 và 4)")
print("đúng :", tree_to_list(Solution().trimBST(build_tree(lst), 1, 3)))
assert tree_to_list(buggy_no_assign(build_tree(lst), 1, 3)) != [3, 2, None, 1]

sai  : [3, 0, 4, None, 2, None, None, 1] (vẫn còn 0 và 4)
đúng : [3, 2, None, 1]


### Lỗi 2 — Đi nhầm hướng khi gốc ra ngoài đoạn

`root.val > high` nghĩa là gốc **quá lớn** ⟹ cây phải càng lớn ⟹ phải sang **trái** (nơi có giá trị nhỏ hơn). Nhiều bạn viết ngược lại vì nhầm "lớn hơn thì đi phải" của phép **tìm kiếm** (nơi ta đi theo giá trị muốn tìm). Ở bài cắt tỉa ta đi về **phía có thể còn nút hợp lệ**. Đã thấy kết quả sai ở mục 12 (`wrong_direction`). Quy tắc nhớ: *gốc to quá thì bỏ nó cùng cây phải; gốc bé quá thì bỏ nó cùng cây trái*.

In [30]:
for lst, low, high in [([3, 0, 4, None, 2, None, None, 1], 1, 3), ([5, 3, 8, 1, 4, 7, 9], 4, 8), ([2, 1, 3], 2, 3)]:
    bad = tree_to_list(wrong_direction(build_tree(lst), low, high))
    good = tree_to_list(Solution().trimBST(build_tree(lst), low, high))
    print(f"{lst} [{low},{high}]: sai = {bad} | đúng = {good}")

[3, 0, 4, None, 2, None, None, 1] [1,3]: sai = [3] | đúng = [3, 2, None, 1]
[5, 3, 8, 1, 4, 7, 9] [4,8]: sai = [5, None, 8, 7] | đúng = [5, 4, 8, None, None, 7]
[2, 1, 3] [2,3]: sai = [2, None, 3] | đúng = [2, None, 3]


### Lỗi 3 — Dựng lại BST cân bằng (hoặc bất kỳ) từ các giá trị còn lại

Giữ đúng tập giá trị **chưa đủ**: đề yêu cầu **giữ cấu trúc tương đối** nên đáp án là một cây **cụ thể**. Dựng cân bằng cho cây khác (mục 12 đã thấy). Thậm chí chèn các giá trị còn lại theo thứ tự **sắp xếp** (thay vì tiền thứ tự) cho cây lệch, cũng sai.

In [31]:
def wrong_insert_sorted(root, low, high):
    new_root = None
    for v in sorted(v for v in collect(root) if low <= v <= high):
        new_root = bst_insert(new_root, v)               # BUG: chèn theo thứ tự tăng -> cây lệch phải
    return new_root

lst, low, high = [5, 3, 8, 1, 4, 7, 9], 4, 8
print("đáp án đúng   :", tree_to_list(Solution().trimBST(build_tree(lst), low, high)))
print("chèn theo sort:", tree_to_list(wrong_insert_sorted(build_tree(lst), low, high)), "(đủ giá trị nhưng sai hình dạng)")

đáp án đúng   : [5, 4, 8, None, None, 7]
chèn theo sort: [4, None, 5, None, 7, None, 8] (đủ giá trị nhưng sai hình dạng)


### Lỗi 4 — Bản lặp quên bước đưa gốc vào đoạn

Nếu bỏ pha 1 và giả định gốc luôn hợp lệ, khi gốc nằm ngoài đoạn thì kết quả sai (giữ lại gốc không hợp lệ). Pha 1 là bắt buộc.

In [32]:
def buggy_no_phase1(root, low, high):
    node = root                                          # BUG: bỏ qua pha 1
    while node:
        while node.left and node.left.val < low:
            node.left = node.left.right
        node = node.left
    node = root
    while node:
        while node.right and node.right.val > high:
            node.right = node.right.left
        node = node.right
    return root

lst, low, high = [3, 0, 4, None, 2, None, None, 1], 4, 9      # gốc 3 < 4
print("sai  :", tree_to_list(buggy_no_phase1(build_tree(lst), low, high)), "(gốc 3 không hợp lệ vẫn còn)")
print("đúng :", tree_to_list(Solution3().trimBST(build_tree(lst), low, high)))
assert tree_to_list(buggy_no_phase1(build_tree(lst), low, high)) != tree_to_list(Solution3().trimBST(build_tree(lst), low, high))

sai  : [3, None, 4] (gốc 3 không hợp lệ vẫn còn)
đúng : [4]


### Lỗi 5 — Đệ quy quá sâu với cây lệch

Chiều cao cây có thể tới `10^4` (cây lệch). Python mặc định giới hạn độ sâu đệ quy khoảng `1000`, nên bản đệ quy có thể báo `RecursionError` trên cây lệch sâu (tùy môi trường chấm có nâng giới hạn hay không). Hãy thử với một cây lệch 5000 nút ở giới hạn mặc định (ta hạ giới hạn xuống thấp để mô phỏng chắc chắn).

In [33]:
chain = tree_to_list(build_bst(range(0, 5000)))
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    Solution2().trimBST(build_tree(chain), 0, 4999)
    print("không lỗi?!")
except RecursionError:
    print("RecursionError với cây lệch 5000 nút ở giới hạn đệ quy 1000")
finally:
    sys.setrecursionlimit(old)

# Bản lặp (Cách 3) và Cách 4 không bị ảnh hưởng
sys.setrecursionlimit(1000)
try:
    r3 = Solution3().trimBST(build_tree(chain), 100, 4900)
    r4 = Solution4().trimBST(build_tree(chain), 100, 4900)
    print("bản lặp / preorder: chạy bình thường, số nút còn lại:", len(preorder_iter(r3)), len(preorder_iter(r4)))
    assert preorder_iter(r3) == preorder_iter(r4) == list(range(100, 4901))
finally:
    sys.setrecursionlimit(old)

RecursionError với cây lệch 5000 nút ở giới hạn đệ quy 1000
bản lặp / preorder: chạy bình thường, số nút còn lại: 4801 4801


## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "chỉnh sửa/cắt tỉa **BST** theo điều kiện về khoảng giá trị" ⇒ dùng tính chất BST để **loại cả cây con** chỉ bằng một phép so sánh.
- Khuôn mẫu đệ quy cây: **`f(node)` trả về gốc mới của cây con**; cha nhận lại bằng `node.left = f(node.left)`.
- Ba trường hợp: nhìn `node.val` so với `low`, `high` ⇒ *quá lớn* (xét trái), *quá bé* (xét phải), *trong đoạn* (giữ và xử lý cả hai con).
- Nếu cần tránh đệ quy: **tìm gốc mới trước**, rồi tỉa hai "xương sống" (chỉ cần `low` ở bên trái, chỉ cần `high` ở bên phải).
- Cách "an toàn" để kiểm tra bài cây: viết `build_tree`/`tree_to_list` và một oracle đơn giản (thu giá trị rồi chèn lại theo tiền thứ tự), sau đó đối chiếu ngẫu nhiên.
- Tiền thứ tự của BST xác định duy nhất cây: dùng khi cần dựng lại cây.
- Các bài cùng họ: **450. Delete Node in a BST**, **938. Range Sum of BST**, **776. Split BST**, **700. Search in a Binary Search Tree**, **98. Validate Binary Search Tree**, **1008. Construct Binary Search Tree from Preorder Traversal**.

## 15. Biến thể

### 15.1. Tổng các giá trị trong đoạn (bài 938 — Range Sum of BST)

Cùng ý tưởng cắt nhánh nhưng không cần sửa cây: nút `< low` chỉ xét cây phải; nút `> high` chỉ xét cây trái; còn lại cộng giá trị và xét cả hai.

In [34]:
def range_sum_bst(root, low, high):
    total, stack = 0, [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if node.val < low:
            stack.append(node.right)
        elif node.val > high:
            stack.append(node.left)
        else:
            total += node.val
            stack.append(node.left)
            stack.append(node.right)
    return total

t = build_tree([10, 5, 15, 3, 7, None, 18])
assert range_sum_bst(t, 7, 15) == 32
assert range_sum_bst(build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6]), 6, 10) == 23
rng = random.Random(30)
for _ in range(300):
    lst = random_bst_list(rng, rng.randint(1, 30), 0, 50)
    low = rng.randint(0, 50)
    high = rng.randint(low, 50)
    assert range_sum_bst(build_tree(lst), low, high) == sum(v for v in collect(build_tree(lst)) if low <= v <= high)
print("range_sum_bst OK:", range_sum_bst(t, 7, 15))

range_sum_bst OK: 32


### 15.2. Đếm số nút nằm trong đoạn và số nút **thực sự thăm** (thấy hiệu quả của cắt nhánh)

Ta đếm cả số nút bị thăm để thấy cắt nhánh tiết kiệm bao nhiêu so với duyệt toàn bộ.

In [35]:
def count_in_range_with_visits(root, low, high):
    visited = count = 0
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        visited += 1
        if node.val < low:
            stack.append(node.right)
        elif node.val > high:
            stack.append(node.left)
        else:
            count += 1
            stack.append(node.left)
            stack.append(node.right)
    return count, visited

rng = random.Random(41)
lst = random_bst_list(rng, 2000, 0, 10 ** 4)
root = build_tree(lst)
for low, high in ((0, 10 ** 4), (4000, 4200), (9990, 10 ** 4), (5000, 5000)):
    cnt, vis = count_in_range_with_visits(build_tree(lst), low, high)
    print(f"[{low:5d},{high:5d}]: {cnt:5d} nút trong đoạn, chỉ thăm {vis:5d} / 2000 nút")
    assert cnt == sum(1 for v in collect(root) if low <= v <= high)

[    0,10000]:  2000 nút trong đoạn, chỉ thăm  2000 / 2000 nút
[ 4000, 4200]:    39 nút trong đoạn, chỉ thăm    49 / 2000 nút
[ 9990,10000]:     1 nút trong đoạn, chỉ thăm    10 / 2000 nút
[ 5000, 5000]:     0 nút trong đoạn, chỉ thăm    11 / 2000 nút


### 15.3. Chỉ cắt một phía (chỉ có `low`, hoặc chỉ có `high`)

Nếu chỉ có cận dưới thì bài còn ngắn hơn: nút `< low` thì thay bằng cây con phải đã xử lý; ngược lại giữ nút và chỉ cần xử lý cây con trái (cây con phải toàn `> node.val >= low`).

In [36]:
def trim_low_only(root, low):
    if root is None:
        return None
    if root.val < low:
        return trim_low_only(root.right, low)
    root.left = trim_low_only(root.left, low)          # cây phải không cần xử lý
    return root

big = 10 ** 9
rng = random.Random(52)
for _ in range(300):
    lst = random_bst_list(rng, rng.randint(1, 30), 0, 50)
    low = rng.randint(0, 50)
    a = tree_to_list(trim_low_only(build_tree(lst), low))
    b = tree_to_list(Solution().trimBST(build_tree(lst), low, big))
    assert a == b
print("trim_low_only trùng với trimBST(low, +vô cùng): OK")
print(tree_to_list(trim_low_only(build_tree([3, 0, 4, None, 2, None, None, 1]), 1)))

trim_low_only trùng với trimBST(low, +vô cùng): OK
[3, 2, 4, 1]


### 15.4. Xóa một nút khỏi BST (bài 450 — cùng khuôn mẫu "trả về gốc mới")

Khuôn mẫu "hàm trả về gốc mới của cây con" cũng giải bài xóa nút. Nếu nút cần xóa có hai con, thay giá trị bằng nút nhỏ nhất của cây phải rồi xóa nút nhỏ nhất đó.

In [37]:
def delete_node(root, key):
    if root is None:
        return None
    if key < root.val:
        root.left = delete_node(root.left, key)
    elif key > root.val:
        root.right = delete_node(root.right, key)
    else:
        if root.left is None:
            return root.right
        if root.right is None:
            return root.left
        succ = root.right
        while succ.left:
            succ = succ.left
        root.val = succ.val
        root.right = delete_node(root.right, succ.val)
    return root

rng = random.Random(63)
for _ in range(400):
    lst = random_bst_list(rng, rng.randint(1, 25), 0, 40)
    vals = collect(build_tree(lst))
    key = rng.randint(0, 40)
    res = delete_node(build_tree(lst), key)
    assert is_bst(res) and collect(res) == [v for v in vals if v != key]
print("delete_node OK (kết quả là BST và còn đúng các giá trị khác key)")
print(tree_to_list(delete_node(build_tree([5, 3, 6, 2, 4, None, 7]), 3)))

delete_node OK (kết quả là BST và còn đúng các giá trị khác key)
[5, 4, 6, 2, None, None, 7]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thu và chèn lại | Thu giá trị, lọc, chèn lại theo tiền thứ tự | `O(n · h)` | `O(n)` | Làm oracle; không dùng khi cây lệch |
| 2. Đệ quy 3 trường hợp (nộp) | Quá lớn → trái, quá bé → phải, trong đoạn → giữ | `O(n)` | `O(h)` | Mặc định: ngắn gọn, dễ chứng minh |
| 3. Lặp + xương sống | Đưa gốc vào đoạn, tỉa xương sống trái và phải | `O(n)` | `O(1)` phụ | Khi cây rất sâu hoặc cần tiết kiệm bộ nhớ |
| 4. Tiền thứ tự cắt nhánh + dựng lại | Duyệt bằng ngăn xếp, dựng cây mới bằng ngăn xếp | `O(n)` | `O(n)` | Khi không được sửa cây gốc, không muốn đệ quy |

Cách 2 và 3 sửa cây tại chỗ; cách 4 tạo cây mới.

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(tree_to_list(trim_recursive(build_tree([2, 1, 3]), 2, 3)))
print(tree_to_list(trim_recursive(build_tree([3, 1, 4, None, 2]), 2, 4)))
print(tree_to_list(trim_recursive(build_tree([5]), 6, 9)))
```

In [38]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `root = [5,3,8,1,4,7,9]`, `low = 4`, `high = 8`: vẽ cây, xác định nút nào bị bỏ, rồi viết ra danh sách `[...]` của cây kết quả và số nút còn lại.

In [39]:
# Bài 2: điền kết quả tính tay
hand_result = None         # list kiểu LeetCode của cây kết quả
hand_remaining = None      # số nút còn lại
hand_removed = None        # số nút bị bỏ

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 2 (đệ quy ba trường hợp).

In [40]:
def exercise_3_recursive(root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
    # TODO 1: nếu root rỗng thì trả None
    # TODO 2: nếu root.val > high thì trả về kết quả cắt tỉa của cây con TRÁI
    # TODO 3: nếu root.val < low thì trả về kết quả cắt tỉa của cây con PHẢI
    # TODO 4: ngược lại gán lại root.left và root.right bằng kết quả cắt tỉa của chúng, rồi trả root
    return None


# assert tree_to_list(exercise_3_recursive(build_tree([1, 0, 2]), 1, 2)) == [1, None, 2]
# assert tree_to_list(exercise_3_recursive(build_tree([3, 0, 4, None, 2, None, None, 1]), 1, 3)) == [3, 2, None, 1]
# assert tree_to_list(exercise_3_recursive(build_tree([5]), 1, 2)) == []

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_iterative(root, low, high)`: pha 1 đưa gốc vào đoạn, pha 2 tỉa xương sống trái, pha 3 tỉa xương sống phải. Không dùng đệ quy.

In [41]:
def exercise_4_iterative(root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
    # TODO
    pass


# assert tree_to_list(exercise_4_iterative(build_tree([1, 0, 2]), 1, 2)) == [1, None, 2]
# assert tree_to_list(exercise_4_iterative(build_tree([5, 3, 8, 1, 4, 7, 9]), 4, 8)) == [5, 4, 8, None, None, 7]
# assert tree_to_list(exercise_4_iterative(build_tree([5]), 1, 2)) == []

### Bài 5 — Mini challenge: số nút còn lại **mà không cần dựng cây mới** (khó)

Viết `exercise_5_count_kept(root, low, high)` trả về số nút sẽ còn lại sau khi cắt tỉa **mà không sửa cây** và **chỉ thăm các nút cần thiết** (cắt nhánh bằng tính chất BST). Gợi ý: nghĩ tới Range Sum nhưng đếm thay vì cộng.

```python
exercise_5_count_kept(build_tree([3, 0, 4, None, 2, None, None, 1]), 1, 3) -> 3
exercise_5_count_kept(build_tree([5]), 1, 2) -> 0
exercise_5_count_kept(build_tree([5, 3, 8, 1, 4, 7, 9]), 4, 8) -> 4
```

In [42]:
def exercise_5_count_kept(root: Optional[TreeNode], low: int, high: int) -> int:
    # TODO
    pass


# assert exercise_5_count_kept(build_tree([3, 0, 4, None, 2, None, None, 1]), 1, 3) == 3
# assert exercise_5_count_kept(build_tree([5]), 1, 2) == 0
# assert exercise_5_count_kept(build_tree([5, 3, 8, 1, 4, 7, 9]), 4, 8) == 4

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: với mỗi gốc, xét ba trường hợp (quá lớn, quá bé, trong đoạn).
- Mức 2: dòng 1: gốc `2` trong `[2,3]`, con trái `1 < 2` bị bỏ, con phải `3` giữ. Dòng 2: gốc `3` giữ; con trái `1 < 2` bị bỏ nhưng con phải của nó (`2`) được nâng lên; con phải `4` giữ.
- Mức 3: dòng 1 là `[2, None, 3]`; dòng 2 là `[3, 2, 4]`; dòng 3 là `[]` (gốc `5 < 6`, đi sang cây phải rỗng).

**Bài 2**
- Mức 1: nút nào nằm ngoài `[4, 8]`? Các nút: `5,3,8,1,4,7,9`.
- Mức 2: bỏ `3`, `1`, `9`. Nút `4` (con phải của `3`) được nâng lên làm con trái của `5`; con trái của `9` rỗng nên `8.right = None`.
- Mức 3: kết quả `[5, 4, 8, None, None, 7]`, `4` nút còn lại, `3` nút bị bỏ.

**Bài 3**
- Mức 1: giống mục 7.1.
- Mức 2: `if root.val > high: return exercise_3_recursive(root.left, low, high)`.
- Mức 3: ở trường hợp giữ, `root.left = ...`, `root.right = ...`, `return root`.

**Bài 4**
- Mức 1: pha 1: `while root and (root.val < low or root.val > high): ...`.
- Mức 2: pha 2: với `node = root`, `while node.left and node.left.val < low: node.left = node.left.right`, rồi `node = node.left`.
- Mức 3: pha 3 đối xứng với `right`, `high`, và `node.right = node.right.left`; nhớ `if root is None: return None` sau pha 1.

**Bài 5**
- Mức 1: dùng ngăn xếp (hoặc đệ quy) như `range_sum_bst`.
- Mức 2: nút `< low` thì chỉ xét con phải; nút `> high` thì chỉ xét con trái; còn lại đếm `1` và xét cả hai con.
- Mức 3: `count += 1` thay cho `total += node.val`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [43]:
ans1 = [
    tree_to_list(trim_recursive(build_tree([2, 1, 3]), 2, 3)),
    tree_to_list(trim_recursive(build_tree([3, 1, 4, None, 2]), 2, 4)),
    tree_to_list(trim_recursive(build_tree([5]), 6, 9)),
]
print(ans1)
assert ans1 == [[2, None, 3], [3, 2, 4], []]

[[2, None, 3], [3, 2, 4], []]


### Đáp án Bài 2

In [44]:
ans2_result = [5, 4, 8, None, None, 7]
ans2_remaining = 4
ans2_removed = 3
res = Solution().trimBST(build_tree([5, 3, 8, 1, 4, 7, 9]), 4, 8)
assert tree_to_list(res) == ans2_result
assert len(preorder_iter(res)) == ans2_remaining
assert 7 - ans2_remaining == ans2_removed
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [45]:
def ans3_recursive(root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
    if root is None:
        return None
    if root.val > high:
        return ans3_recursive(root.left, low, high)
    if root.val < low:
        return ans3_recursive(root.right, low, high)
    root.left = ans3_recursive(root.left, low, high)
    root.right = ans3_recursive(root.right, low, high)
    return root

assert tree_to_list(ans3_recursive(build_tree([1, 0, 2]), 1, 2)) == [1, None, 2]
assert tree_to_list(ans3_recursive(build_tree([3, 0, 4, None, 2, None, None, 1]), 1, 3)) == [3, 2, None, 1]
assert tree_to_list(ans3_recursive(build_tree([5]), 1, 2)) == []
rng = random.Random(71)
for _ in range(300):
    lst = random_bst_list(rng, rng.randint(1, 30), 0, 50)
    low = rng.randint(0, 50)
    high = rng.randint(low, 50)
    assert tree_to_list(ans3_recursive(build_tree(lst), low, high)) == tree_to_list(trim_brute(build_tree(lst), low, high))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [46]:
def ans4_iterative(root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
    while root and (root.val < low or root.val > high):
        root = root.right if root.val < low else root.left
    if root is None:
        return None
    node = root
    while node:
        while node.left and node.left.val < low:
            node.left = node.left.right
        node = node.left
    node = root
    while node:
        while node.right and node.right.val > high:
            node.right = node.right.left
        node = node.right
    return root

assert tree_to_list(ans4_iterative(build_tree([1, 0, 2]), 1, 2)) == [1, None, 2]
assert tree_to_list(ans4_iterative(build_tree([5, 3, 8, 1, 4, 7, 9]), 4, 8)) == [5, 4, 8, None, None, 7]
assert tree_to_list(ans4_iterative(build_tree([5]), 1, 2)) == []
rng = random.Random(72)
for _ in range(300):
    lst = random_bst_list(rng, rng.randint(1, 30), 0, 50)
    low = rng.randint(0, 50)
    high = rng.randint(low, 50)
    assert tree_to_list(ans4_iterative(build_tree(lst), low, high)) == tree_to_list(trim_brute(build_tree(lst), low, high))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [47]:
def ans5_count_kept(root: Optional[TreeNode], low: int, high: int) -> int:
    count, stack = 0, [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if node.val < low:
            stack.append(node.right)
        elif node.val > high:
            stack.append(node.left)
        else:
            count += 1
            stack.append(node.left)
            stack.append(node.right)
    return count

assert ans5_count_kept(build_tree([3, 0, 4, None, 2, None, None, 1]), 1, 3) == 3
assert ans5_count_kept(build_tree([5]), 1, 2) == 0
assert ans5_count_kept(build_tree([5, 3, 8, 1, 4, 7, 9]), 4, 8) == 4
rng = random.Random(73)
for _ in range(300):
    lst = random_bst_list(rng, rng.randint(1, 30), 0, 50)
    low = rng.randint(0, 50)
    high = rng.randint(low, 50)
    kept = len(preorder_iter(Solution3().trimBST(build_tree(lst), low, high)))
    assert ans5_count_kept(build_tree(lst), low, high) == kept
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def trimBST(self, root: Optional[TreeNode], low: int, high: int) -> Optional[TreeNode]:
        def dfs(node):
            if node is None:
                return None
            if node.val > high:              # quá lớn: bỏ nó và cây phải -> xét cây trái
                return dfs(node.left)
            if node.val < low:               # quá bé: bỏ nó và cây trái -> xét cây phải
                return dfs(node.right)
            node.left = dfs(node.left)       # trong đoạn: giữ, gán lại hai con
            node.right = dfs(node.right)
            return node

        return dfs(root)
```

Nhớ ba ý: **BST cho phép loại cả cây con bằng một phép so sánh**, **hàm trả về gốc mới và cha phải gán lại**, **cây lệch sâu thì dùng bản lặp (tìm gốc mới rồi tỉa hai xương sống)**.

### Checklist trước khi submit

- [ ] Xử lý `root is None` (và kết quả có thể là cây rỗng `None`).
- [ ] Hướng đi đúng: `val > high` → trái, `val < low` → phải.
- [ ] Đã **gán lại** `node.left = dfs(node.left)` và `node.right = dfs(node.right)`.
- [ ] Hiểu rằng đáp án giữ nguyên cấu trúc tương đối (không dựng lại cây cân bằng).
- [ ] Cân nhắc độ sâu đệ quy với cây lệch `10^4` nút (hoặc dùng bản lặp).
- [ ] Đã thử: một nút ngoài đoạn, một nút đúng biên, đoạn phủ hết cây, đoạn nằm ngoài cây, gốc bị loại.
- [ ] Time `O(n)`, Space `O(h)` (đệ quy) hoặc `O(1)` (lặp).